# Serve the fine-tuned planner with vLLM (Colab)

Serves **base model + LoRA adapter** through vLLM's OpenAI-compatible API, then exposes it with a temporary
Cloudflare tunnel so the search engine on your laptop can call it.

- Base: `unsloth/Qwen3.8-27B-unsloth-bnb-4bit` (the exact weights the adapter was trained on), ~17 GB → fits an A100 40 GB.
- Adapter: from Google Drive (saved by the fine-tuning notebook) or the Hugging Face Hub.
- Structured output: the engine sends `response_format = json_schema`; vLLM enforces it with guided decoding.

Qwen3.5/3.8 need a recent vLLM (Unsloth notes 0.16 lacks support). If the install below can't load the
architecture, switch `VLLM_SPEC` to the nightly line.

In [ ]:
PROFILE = "qwen3.8-27b-qlora"   # or "qwen3.5-4b-smoke" to test on a T4
ADAPTER_SOURCE = "drive"         # "drive" | "hf"
HF_REPO = ""                     # used when ADAPTER_SOURCE == "hf"
SERVED_NAME = "fedsearch-qwen3.8-27b"
VLLM_SPEC = '"vllm>=0.17"'       # nightly: "vllm --pre --extra-index-url https://wheels.vllm.ai/nightly"
PORT = 8000

In [ ]:
import os, subprocess
REPO = "priyankavlt/platform-agnostic-federated-search-with-fine-tuned-llm"
BRANCH = "main"
WORKDIR = "/content/fedsearch"

def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

if not os.path.exists(WORKDIR):
    token = _secret("GH_TOKEN")  # only needed for a private fork
    url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, WORKDIR], check=True)
os.chdir(WORKDIR)
# An editable pip install isn't visible to an already-running kernel, so put the code on the path directly.
import sys
sys.path[:0] = [f"{WORKDIR}/src", WORKDIR]
os.environ["PYTHONPATH"] = f"{WORKDIR}/src:{WORKDIR}"
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
%%capture
!pip install {VLLM_SPEC} bitsandbytes
!pip install --no-deps -e /content/fedsearch
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared

In [ ]:
import yaml
from pathlib import Path
prof = yaml.safe_load(Path("configs/train_qlora.yaml").read_text())["profiles"][PROFILE]
BASE = prof["base_model"]

if ADAPTER_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    ADAPTER = f"/content/drive/MyDrive/fedsearch/{Path(prof['output_dir']).name}/adapter"
else:
    from huggingface_hub import snapshot_download
    ADAPTER = snapshot_download(HF_REPO, token=_secret("HF_TOKEN"))
print("base:", BASE, "\nadapter:", ADAPTER, Path(ADAPTER).exists())

## Start vLLM (background)

In [ ]:
import subprocess, time, urllib.request
quant = ["--quantization", "bitsandbytes"] if prof["load_in_4bit"] else []
cmd = ["vllm", "serve", BASE, *quant, "--served-model-name", "base",
       "--enable-lora", "--max-lora-rank", str(prof["lora_r"]), "--lora-modules", f"{SERVED_NAME}={ADAPTER}",
       "--max-model-len", "4096", "--gpu-memory-utilization", "0.90", "--port", str(PORT),
       "--enable-auto-tool-choice", "--tool-call-parser", "hermes"]
print(" ".join(cmd))
log = open("/content/vllm.log", "w")
server = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)

for _ in range(180):  # model download + load can take ~10 min the first time
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/v1/models", timeout=2); print("vLLM is up"); break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError("vLLM exited:\n" + open("/content/vllm.log").read()[-4000:])
        time.sleep(5)
else:
    print(open("/content/vllm.log").read()[-3000:])

## Sanity check: schema-constrained QueryPlan from the fine-tuned adapter

In [ ]:
from openai import OpenAI
from fedsearch.schemas import QueryPlan
from fedsearch.search.prompts import PLANNER_SYSTEM

client = OpenAI(base_url=f"http://localhost:{PORT}/v1", api_key="EMPTY")
resp = client.chat.completions.create(
    model=SERVED_NAME, temperature=0,
    messages=[{"role": "system", "content": PLANNER_SYSTEM},
              {"role": "user", "content": "is globx still having trouble with okta login?"}],
    response_format={"type": "json_schema", "json_schema": {"name": "QueryPlan", "schema": QueryPlan.model_json_schema()}},
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)
print(QueryPlan.model_validate_json(resp.choices[0].message.content).model_dump_json(indent=1))

## Expose it to your laptop

In [ ]:
import re
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    if m := re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line):
        URL = m.group(0); break
print(f"""
On your laptop:

  export FEDSEARCH_PLANNER=llm FEDSEARCH_ANSWERER=llm
  export FEDSEARCH_LLM_BASE_URL={URL}/v1 FEDSEARCH_LLM_MODEL={SERVED_NAME}
  uv run fedsearch search "is globx still having trouble with okta login?"
  uv run fedsearch eval                               # fine-tuned
  FEDSEARCH_LLM_MODEL=base uv run fedsearch eval      # base model, same server

The tunnel lives as long as this Colab session. Anyone with the URL can call it; stop it when done.
""")

In [ ]:
# Stop everything
tunnel.terminate(); server.terminate()